In [12]:
config = {
    "model_type": "skip-light",
    "input_shape": 4,
    "amplitude_shape": 36,
    "data_dir": "/scratch/akula.ha/dataset",
    "seed": 42,
    "var_y": [
        "y5",
        "y6"
    ],
    "activation": "leaky_relu",
    "width": 31,
    "depth": 10,
    "skip_block_layers": 5,
    "beta": 0,
    "alpha": 0,
    "normal_scaled": False,
    "lr_decay_type": "exp",
    "initial_lr": 0.003,
    "final_lr": 1e-06,
    "decay_steps": 120000,
    "train-sample-size": 10000000,
    "validate-sample-size": 500000,
    "test-sample-size": 500000,
    "use_MC_sample": False,
    "batch_size": 1024,
    "steps_per_epoch": 2400,
    "early_stopping_start_epoch": 50,
    "patience": 50,
    "monitor": "val_mse",
    "loss": "mse",
    "gradient_clipping": True,
    "verbose": 1,
    "base_directory": "../models/",
    "epochs": 2000,
    "model-uuid": "ef8e7636",
    "start_time": "2025-05-07 21:39:40 -0400",
    "device": "cuda",
    "directory": "../models/skip-light-torch-4D-y5_y6-ef8e7636",
    "train-samples-used": 6,
    "validate-samples-used": 6,
    "test-samples-used": 6,
    "n_targets": 2,
    "scaling": {},
    "trainable_parameters": 49819,
    "non_trainable_parameters": 0,
    "total_parameters": 49819,
    "final_decayed_lr": 4.4758200183300384e-05,
    "final_validation_r2": 99.74279299379268,
    "final_validation_loss": 0.026469253109972397,
    "last_epoch": 182,
    "n_training_epochs": 132,
    "fit_time": "2:05:48",
    "test_metrics": {
        "r2": {
            "y5": 59.34949379940995,
            "y6": 41.53556692441772,
            "model": 99.8367708984802
        },
        "abs_score": {
            "y5": 99.47024307869438,
            "y6": 99.65569012719476,
            "model": 97.01768156003165
        }
    },
    "end_time": "2025-05-07 23:47:09 -0400"
}

In [13]:
import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt
from pyspark.sql import SparkSession
from pyspark.sql import dataframe
from pyspark.sql.types import StructType, StructField, DoubleType
from sklearn import metrics
import time
import sys
import logging
import uuid
import os
import shutil
import json
import argparse
import pytorch_model_summary as pms

In [14]:
def init_torch(config):
    ''' function for initializing the seeds
        arguments:
            config: the configuration file
    '''
    gen = torch.manual_seed(config['seed'])
    torch.set_default_dtype(torch.float64)
    
    device = get_device()

    if device.type == 'cpu':
        torch.set_num_threads(64)
        # torch.set_num_interop_threads(1)

    logging.info(f' torch is using {device}')
    
    return device

In [15]:
def get_device():
    ''' function to get the device the NN is running on, CPU or GPU
    '''
    if torch.cuda.is_available():
        device = torch.device("cuda:0")
    else: 
        device = torch.device("cpu")
        
    return device

In [16]:
device = init_torch(config)

In [17]:
import re

In [18]:
def load_data(config):
    """ Load the data using pyspark
        arguments:
            config: the configuration file
        returns:
            df['train']: the training dataframe
            df['validate']: the validation data set
            df['test']: the testing data set
            spark: the spark session
    """
    
    # Spark session and configuration
    logging.info(' creating Spark session')
    spark = (SparkSession.builder.master("local[48]")
             .config('spark.executor.instances', 16)
             .config('spark.executor.cores', 16)
             .config('spark.executor.memory', '10g')
             .config('spark.driver.memory', '15g')
             .config('spark.memory.offHeap.enabled', True)
             .config('spark.memory.offHeap.size', '20g')
             .config('spark.dirver.maxResultSize', '20g')
             .config('spark.debug.maxToStringFields', 100)
             .appName("amp.hell").getOrCreate())

    # Enable Arrow-based columnar data 
    spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")
    spark.conf.set(
        "spark.sql.execution.arrow.pyspark.fallback.enabled", "true"
    )
    logging.info(' Spark initialized')
    
    # read the data into a spark frame
    start = time.time()
    path = config['data_dir']
    if path[-1] != '/':
        path = path + '/'
    input_var = ['x'+str(i+1) for i in range(config['input_shape'])]
    target_var = ['y'+str(i+1) for i in range(config['amplitude_shape'])]

    header = input_var + target_var
    schema = StructType([StructField(header[i], DoubleType(), True) for i in range(config['input_shape']+config['amplitude_shape'])])

    df = {}
    df['train'] = spark.read.options(delimiter=',').schema(schema).format("csv").load(path+'train/*.csv.*', header='true')
    
    
    if config['var_y'] == 'all': config['var_y'] = df['train'].columns[config['input_shape']:]
    
    
    if config['use_MC_sample']:
        mc_suffix = '_mc'
    else:
        mc_suffix = ''
    df['validate'] = spark.read.options(delimiter=',').schema(schema).format("csv").load(path+'validate'+mc_suffix+'/*.csv.*', header='true')
    df['test'] = spark.read.options(delimiter=',').schema(schema).format("csv").load(path+'test'+mc_suffix+'/*.csv.*', header='true')

    logging.info(' data loaded into Spark session in {:.3f} seconds'.format(time.time() - start))
    
    # transfer the data to a pandas dataframe
    start = time.time()
    train_sample = min(config['train-sample-size'], df['train'].count())
    validate_sample = min(config['validate-sample-size'], df['validate'].count())
    test_sample = min(config['test-sample-size'], df['test'].count())
    
    ########################################################################
    # Assuming config['var_y'] is the list
    operators = ['+', '-', '*', '/']

    # List of items with operators
    with_operator = [item for item in config['var_y'] if any(op in item for op in operators)]
    print(with_operator)
    # List of items without operators
    without_operator = [item for item in config['var_y'] if all(op not in item for op in operators)]
    print(without_operator)
    
    # Split the items with operators into individual components
    split_items = [subitem for item in with_operator for subitem in re.split(r'\+|\-|\*|\/', item)]

    # Combine the split items with the items without operators
    combined_list = without_operator + split_items

    # Remove duplicates to get a unique list
    unique_list = list(set(combined_list))

    # Sort the list if you want it in a specific order (optional)
    unique_list.sort()
    
    ########################################################################
    
    df['train'] = df['train'].select(*input_var, *unique_list).limit(train_sample).toPandas() 
    df['validate'] = df['validate'].select(*input_var, *unique_list).limit(validate_sample).toPandas()
    df['test'] = df['test'].select(*input_var, *unique_list).limit(test_sample).toPandas()
    
    # Step 1: Perform the operations
    for key in ['train', 'validate', 'test']:
        for expr in config['var_y']:
            if any(op in expr for op in ['+', '-', '*', '/']):
                # Split the expression into individual components and operator
                components = re.split(r'(\+|\-|\*|\/)', expr)

                # Evaluate the expression and create a new column
                if len(components) == 3:  # This should match the pattern "y1 + y3"
                    col1, operator, col2 = components

                    if operator == '+':
                        df[key][expr] = df[key][col1] + df[key][col2]
                    elif operator == '-':
                        df[key][expr] = df[key][col1] - df[key][col2]
                    elif operator == '*':
                        df[key][expr] = df[key][col1] * df[key][col2]
                    elif operator == '/':
                        df[key][expr] = df[key][col1] / df[key][col2]

    # Filter split_items to keep only those that are also in without_operator
    filtered_items = [item for item in split_items if item not in without_operator]

    # Drop the filtered items from the DataFrames
    for key in ['train', 'validate', 'test']:
        df[key] = df[key].drop(columns=filtered_items, errors='ignore')

    
    logging.info(' training data shape: {} x {}'.format(df['train'].shape[0], df['train'].shape[1]))
    logging.info(' validation data shape: {} x {}'.format(df['validate'].shape[0], df['validate'].shape[1]))
    logging.info(' testing data shape: {} x {}'.format(df['test'].shape[0], df['test'].shape[1]))
    
    config['train-samples-used'] = df['train'].shape[1]
    config['validate-samples-used'] = df['validate'].shape[1]
    config['test-samples-used'] = df['test'].shape[1]

    logging.info(' data loaded into pandas dataframe in {:.3f} seconds'.format(time.time() - start))
    
    logger = spark._jvm.org.apache.log4j
    logging.getLogger("py4j.clientserver").setLevel(logging.WARN)

    return df, spark


In [20]:
df, spark = load_data(config)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/05/08 15:36:48 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


[]
['y5', 'y6']


In [21]:
df

{'train':                    x1        x2        x3        x4          y5          y6
 0        5.308962e-08  0.309913  0.050721  0.629043  119.890134 -131.188649
 1        3.505005e-07  0.281701  0.997791  0.482102  -11.651600   15.257625
 2        4.480013e-07  0.609822  0.147178  0.852029  227.064607 -242.366660
 3        6.298025e-07  0.258704  0.408208  0.122103  -27.431008   31.266244
 4        1.014703e-06  0.417312  0.940159  0.743944  111.235489 -114.351289
 ...               ...       ...       ...       ...         ...         ...
 3393721  9.937817e-01  0.520471  0.116823  0.381197   -3.462783   -0.628595
 3393722  9.970094e-01  0.248280  0.539416  0.161634   -5.912596    0.031616
 3393723  9.974963e-01  0.255944  0.522281  0.471605   -7.472055   -0.181993
 3393724  9.976171e-01  0.892835  0.290839  0.270442   -3.183277   -1.728482
 3393725  9.990412e-01  0.747992  0.462169  0.535807   -3.356216   -1.160603
 
 [3393726 rows x 6 columns],
 'validate':                   x1   

In [22]:
def build_data_loaders(df, config):
    ''' function to build the data loaders
        arguments:
            df: dictionary of pandas dataframe for test, validation, and train
            config: configuration file
        returns:
            train_loader: the training set loader
            validation_loader: the validation set loader
            test_loader: the test set loader
    '''
    
    # slice and dice the data into train, validation and test sets
    df_test_data =  df['test']
    df_validation_data = df['validate']
    df_train_data = df['train']

    test_data = df_to_tensor(df_test_data, config)
    validation_data = df_to_tensor(df_validation_data, config)
    train_data = df_to_tensor(df_train_data, config)

    # load the data into torch tensor batches
    batch = config["batch_size"]
    numworkers = 2 if get_device().type == 'cpu' else 0
    train_loader = torch.utils.data.DataLoader(train_data, batch_size=batch, shuffle=True, num_workers=numworkers, drop_last=True)
    validation_loader = torch.utils.data.DataLoader(validation_data, batch_size=batch, shuffle=True, num_workers=numworkers, drop_last=True)
    test_loader = torch.utils.data.DataLoader(test_data, batch_size=batch, shuffle=False, num_workers=numworkers, drop_last=True)
    
    return train_loader, validation_loader, test_loader


In [36]:
len(train_loader)

NameError: name 'train_loader' is not defined

In [37]:

class df_to_tensor(torch.utils.data.Dataset):
    ''' class to convert dataframe to torch tensor
    '''
 
    def __init__(self, df, config):
        self.df = df.copy(deep = True)
        
        # extract x and scale
        x = df.iloc[:, :config['input_shape']]
        df['x1'] = df['x1'].apply(lambda x: x_scale(x))
        
        # extract y
        # if config['var_y'] == 'all':
        #     y = df.iloc[:, config['input_shape']:]
        # else:
        y = df[config['var_y']]
        
        # number of targets
        config['n_targets'] = y.shape[1]
            
        # scale y
        config['scaling'] = {}
        for column in y.columns:
            y[column] = y[column].apply(lambda y: y_scale(y))
            if config['normal_scaled']:
                y[column] = normalize(y, config, column)

        # put them in tensors. Note: the reshaping of y.
        self.x = torch.tensor(x.values, dtype=torch.float64).to(get_device())
        self.y = torch.tensor(y.values, dtype=torch.float64).reshape(-1, config['n_targets']).to(get_device())
 
    def __len__(self):
        return len(self.x)
   
    def __getitem__(self,idx):
        return self.x[idx], self.y[idx]
    
    

In [38]:
def normalize(df, config, var_y):
    """ a function to normaliza the target distribution
        arguments:
            df: dataframe containing the target variable
            config: the config file with the run configuration
            var_y: the variable to be normalized
    """
    config['scaling'][var_y] = {}
    config['scaling'][var_y]["mu"] = df[var_y].mean()
    config['scaling'][var_y]["sigma"] = df[var_y].std()
    
    return (df[var_y] - config['scaling'][var_y]["mu"])/config['scaling'][var_y]["sigma"]


def x_scale(x, p=7.5):
    ''' function for scaling x1
        argument:
            x: the input variable
            p: the scaling factor (default: 7.5)
        returns:
            the scaled variable
    '''
    return 1/p * np.log(1 + x * (np.exp(p) - 1))
                        
    
def y_scale(y):
    ''' function for scaling x1
        argument:
            y: the input variable
        returns:
            the scaled variable
    '''
    return np.log(1 + y) if y >= 0 else -np.log(1 - y)


def y_unscale(y):
    ''' function for scaling x1
        argument:
            y: the input variable
        returns:
            the scaled variable
    '''
    return np.exp(y) - 1 if y >= 0 else 1 - np.exp(-y)

In [39]:
df

{'train':                    x1        x2        x3        x4          y5          y6
 0        5.308962e-08  0.309913  0.050721  0.629043  119.890134 -131.188649
 1        3.505005e-07  0.281701  0.997791  0.482102  -11.651600   15.257625
 2        4.480013e-07  0.609822  0.147178  0.852029  227.064607 -242.366660
 3        6.298025e-07  0.258704  0.408208  0.122103  -27.431008   31.266244
 4        1.014703e-06  0.417312  0.940159  0.743944  111.235489 -114.351289
 ...               ...       ...       ...       ...         ...         ...
 3393721  9.937817e-01  0.520471  0.116823  0.381197   -3.462783   -0.628595
 3393722  9.970094e-01  0.248280  0.539416  0.161634   -5.912596    0.031616
 3393723  9.974963e-01  0.255944  0.522281  0.471605   -7.472055   -0.181993
 3393724  9.976171e-01  0.892835  0.290839  0.270442   -3.183277   -1.728482
 3393725  9.990412e-01  0.747992  0.462169  0.535807   -3.356216   -1.160603
 
 [3393726 rows x 6 columns],
 'validate':                   x1   

In [40]:
train_loader, validation_loader, test_loader = build_data_loaders(df,config)

/tmp/ipykernel_2387537/3488228483.py:24: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  y[column] = y[column].apply(lambda y: y_scale(y))
/tmp/ipykernel_2387537/3488228483.py:24: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  y[column] = y[column].apply(lambda y: y_scale(y))
/tmp/ipykernel_2387537/3488228483.py:24: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pa

In [41]:
t = list(enumerate(test_loader))[:3]

In [42]:
list(test_loader)

[[tensor([[5.4344e-07, 1.4743e-01, 7.7435e-01, 2.4598e-01],
          [2.2544e-06, 7.0544e-01, 5.1185e-01, 3.4134e-02],
          [3.1350e-06, 3.4823e-01, 4.6095e-01, 2.8326e-01],
          ...,
          [9.4767e-04, 7.8192e-01, 8.3783e-01, 9.4324e-01],
          [9.4998e-04, 7.7186e-01, 4.4098e-01, 3.3074e-01],
          [9.5204e-04, 8.9077e-01, 5.2009e-01, 5.5625e-01]], device='cuda:0'),
  tensor([[-5.1674,  5.2362],
          [ 5.0330, -5.0694],
          [ 4.1689, -4.1843],
          ...,
          [ 4.9004, -5.0683],
          [ 4.7817, -5.0497],
          [ 4.6619, -4.6458]], device='cuda:0')],
 [tensor([[9.5312e-04, 5.5020e-01, 6.4033e-01, 9.4720e-01],
          [9.5428e-04, 6.5420e-01, 5.1618e-01, 3.6859e-01],
          [9.5452e-04, 8.7629e-01, 6.2135e-01, 6.7370e-01],
          ...,
          [3.4149e-03, 4.0643e-01, 9.9088e-01, 3.1066e-01],
          [3.4152e-03, 2.4816e-01, 2.5298e-01, 3.1006e-02],
          [3.4161e-03, 8.4951e-01, 9.6366e-01, 3.5161e-01]], device='cuda:0'

In [43]:
def test_model(model, test_data, config):
    ''' function for testing the model
        arguments:
            model: the pytorch model to nbe trained
            test_data: the test data loader
            config: the configurations file
        returns:
            the absolute error
            the R2 score
    '''
    
    for i, data in enumerate(test_data):
        
        # forward prop
        x, y = data
        y_p = model(x)
        
        # store data
        if i == 0:
            x_test = x.cpu().detach().numpy()
            y_test = y.cpu().detach().numpy()
            y_pred = y_p.cpu().detach().numpy()
        else: 
            x_test = np.vstack((x_test, x.cpu().detach().numpy()))
            y_test = np.vstack((y_test, y.cpu().detach().numpy()))
            y_pred = np.vstack((y_pred, y_p.cpu().detach().numpy()))
        
    
    # accuracy
    y_test = np.array(y_test).reshape(-1, config['n_targets'])
    y_pred = np.array(y_pred).reshape(-1, config['n_targets'])
    abs_score = (1 - np.mean(np.abs((y_pred - y_test)/y_test)))*100
    r2_score = metrics.r2_score(y_test, y_pred)*100
    
    # save test results
    df_pred = pd.DataFrame(x_test, columns=['x'+str(i+1) for i in range(config['input_shape'])])
    
    # unscale the target
    scaled_cname = [s + '_scaled' for s in config['var_y']]
    df_pred = pd.concat([df_pred, pd.DataFrame(y_test, columns=scaled_cname)], axis=1)
    for col in df_pred.columns[4:]:
        df_pred[col[:-7]] = df_pred[col].apply(lambda y: y_unscale(y))
    # y_test_real = pd.DataFrame(np.vectorize(y_unscale)(y_test), columns=config['var_y']) # 
    # df_pred = pd.concat([df_pred, y_test_real], axis=1)
    y_test_real = df_pred.iloc[:,-config['n_targets']:].to_numpy()
    
    # stuff in the scaled predictions
    pred_cname = [s + '_scaled_pred' for s in config['var_y']]
    df_pred = pd.concat([df_pred, pd.DataFrame(y_pred, columns=pred_cname)], axis=1)
    
    # stuff in the unscaled predictions
    pred_cname = [s + '_pred' for s in config['var_y']]
    for col in df_pred.columns[-config['n_targets']:]:
        df_pred[col[:-12]+'_pred'] = df_pred[col].apply(lambda y: y_unscale(y))
    # y_pred_real = pd.DataFrame(np.vectorize(y_unscale)(y_pred), columns=pred_cname) # .apply(lambda y: y_unscale(y))
    # df_pred = pd.concat([df_pred, y_pred_real], axis=1)
    y_pred_real = df_pred.iloc[:,-config['n_targets']:].to_numpy()
    
    # calculate the deltas for the scaled and unscaled targets
    scaled_delta_cname = ['scaled_delta_' + s for s in config['var_y']]
    df_pred = pd.concat([df_pred, pd.DataFrame((y_pred - y_test)/y_test*100, columns=scaled_delta_cname)], axis=1)
    delta_cname = ['delta_' + s for s in config['var_y']]
    df_pred = pd.concat([df_pred, pd.DataFrame((y_pred_real - y_test_real)/y_test_real*100, columns=delta_cname)], axis=1)
    
    #same the whole dataframe
    #df_pred.to_csv(config['directory']+'/test-results-'+config['model-uuid']+'-'+f'{abs_score:.6f}-{r2_score:.6f}.csv')
    
    # plot all the scaled and real deltas
    """for c in scaled_delta_cname:
        make_error_plot(config, df_pred, col=c)
    for c in delta_cname:
        make_error_plot(config, df_pred, col=c)"""
        
    config['test_metrics'] = {}
    config['test_metrics']['r2'] = {}
    config['test_metrics']['abs_score'] = {}
    num_vars = len(config['var_y'])
    base_length = config['input_shape'] + num_vars
    for i in range(base_length, base_length + num_vars):
        test = df_pred.iloc[:,i].values
        pred = df_pred.iloc[:,i + 2 * num_vars].values
        config['test_metrics']['r2'][config['var_y'][i - base_length]] = metrics.r2_score(test, pred)*100
        config['test_metrics']['abs_score'][config['var_y'][i - base_length]] = 100 - np.abs(df_pred.iloc[:,i + 4 * num_vars].mean())
    
    config['test_metrics']['r2']['model'] = r2_score
    config['test_metrics']['abs_score']['model'] = abs_score
    
    return abs_score, r2_score

## Model classes

In [24]:
class EarlyStopping:
    ''' class for early stopping with patience
    '''
    def __init__(self, m_path, patience=1, min_delta=0):
        '''
            arguments:
                m_path: the model path where the checkpoints go
                patience: the number of epochs the patience lasts
                min_delta: the minimum change that is monitored
        '''
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.min_validation_loss = np.inf
        self.m_path = m_path
        
    def reset_counter(self):
        ''' function to reset the counter
        '''
        self.counter = 0

    def early_stop(self, model, validation_loss, epoch, config):
        ''' function to check for the early stopping threshold
            arguments:
                model: the torch model
                validation_loss: the validation loss
        '''
        if epoch > config['early_stopping_start_epoch']: 
            if validation_loss < self.min_validation_loss:
                self.min_validation_loss = validation_loss
                torch.save(model.state_dict(), self.m_path)
                config['n_training_epochs'] = epoch
                self.counter = 0
            elif validation_loss > (self.min_validation_loss + self.min_delta):
                self.counter += 1
                if self.counter >= self.patience:
                    return True
            return False
        else:
            return False
    
    
class skip_block(torch.nn.Module):
    """ the basic building block of a dnn with skip connections
        arguments:
            x: the input
            width: the width of the hidden layers
            activation: the activation function: 'relu', 'elu', 'swish' (silu), 'leaky_relu', 'softplus'
            squeeze: a boolean specifying wheher the skip units are squeezed
        returns:
            res: the skip net block
    """
    def __init__(self, input_shape, width, activation, stream = False, n_layers=1):
        ''' init function
            arguments:
                input_shape: the input shape of the block
                width: the width of the block
                activation: the activation function
                stream: build a residual stream or not (default: false)
                n_layers: the number of layers in the block (default: 1)
        '''
        super(skip_block, self).__init__()
        self.input_shape = input_shape
        self.width = width
        self.stream = stream
        
        self.input = torch.nn.Linear(self.input_shape, self.width)
        self.fc_module = torch.nn.ModuleList([torch.nn.Linear(self.width, self.width) for i in range(n_layers)])
        
        if self.stream:
            self.linear = torch.nn.Linear(self.width, self.input_shape)
        else:    
            self.linear = torch.nn.Linear(self.width, self.width)
            
        if self.input_shape != self.width:
            self.reshape = torch.nn.Linear(self.input_shape, self.width, bias=False)
        
        self.act = activation
        
    def forward(self, x):
        ''' forward propagation
        '''
        y = self.act(self.input(x))
        for l in self.fc_module:
            y = self.act(l(y))
        if self.stream:
            y = self.act(self.linear(y))
            y += x
            return y
        else:
            y = self.linear(y)
            if self.input_shape != self.width:
                residual = self.reshape(x)
            else:
                residual = x
            y += residual

            return self.act(y)
        
        
def getActivation(config):
    ''' function for defining the activation function
        argument:
            config: the configurations file
        returns
            the specified activations function
    '''
    if config["activation"] == 'leaky_relu':
        return torch.nn.LeakyReLU()
    elif config["activation"] == 'relu':
        return torch.nn.ReLU()
    if config["activation"] == 'softplus':
        return torch.nn.Softplus()
    if config["activation"] == 'swish':
        return torch.nn.SiLU()
    if config["activation"] == 'sigmoid':
        return torch.nn.Sigmoid()
        
    
class skip_dnn(torch.nn.Module):
    ''' class for the DNN with skip connections: see https://arxiv.org/abs/2302.00753
    '''
    def __init__(self, sk_block, config, stream = False):
        super(skip_dnn, self).__init__()
        self.width = config["width"]
        self.n_blocks = config["depth"] - 1
        self.input_shape = config["input_shape"]
        self.output_shape = config['n_targets']
        self.n_layers = config['skip_block_layers']
        self.stream = stream
        self.act = getActivation(config)
        
        self.input = skip_block(self.input_shape, 
                                self.width, 
                                self.act, 
                                stream = self.stream, 
                                n_layers = self.n_layers)
        self.core = self.make_layers(skip_block)
        if self.stream: 
            self.output = torch.nn.Linear(self.input_shape, self.output_shape)
        else: 
            self.output = torch.nn.Linear(self.width, self.output_shape)
            
    def make_layers(self, skip_block):
        layers = []
        for bl in range(self.n_blocks):
            if self.stream: 
                layers.append(skip_block(self.input_shape, 
                                         self.width, 
                                         self.act, 
                                         stream=self.stream, 
                                         n_layers = self.n_layers))
            else:
                layers.append(skip_block(self.width, 
                                         self.width, 
                                         self.act, 
                                         n_layers = self.n_layers))
            
        return torch.nn.Sequential(*layers)
        
    def forward(self, x):
        x = self.input(x)
        x = self.core(x)
        return self.output(x)
    
    
class dnn(torch.nn.Module):
    ''' class for a feed-forward DNN
    '''
    def __init__(self, config):
        super(dnn, self).__init__()
        self.width = config["width"]
        
        self.input = torch.nn.Linear(config["input_shape"], self.width)
        self.fc_module = torch.nn.ModuleList([torch.nn.Linear(self.width, self.width) for i in range(config["depth"] - 1)])
        self.output = torch.nn.Linear(self.width, config['n_targets'])
        self.act = getActivation(config)
        
    def forward(self, x):
        x = self.act(self.input(x))
        for l in self.fc_module:
            x = self.act(l(x))
        return self.output(x)



class skip_light_module(torch.nn.Module):
    """ 
        This is a skip dnn component of the skip_light neural network 
        argument:
            config: the configurations file
    """
    def __init__(self, config):
        super(skip_light_module, self).__init__()
        self.width = config["width"]
        self.skip_layer_depth = config["skip_block_layers"]
        self.act = getActivation(config)
        self.fc_module = torch.nn.ModuleList([torch.nn.Linear(self.width, self.width) 
                                              for i in range(self.skip_layer_depth)])
        # Apply Xavier (Glorot) normal initialization to layers of fc_module
        for layer in self.fc_module:
            torch.nn.init.xavier_normal_(layer.weight)
            torch.nn.init.zeros_(layer.bias)
        
    def forward(self, x):
        y = x
        # vector x shape and width are the same
        for layer in self.fc_module:
            y = self.act(layer(y))
        y = y+x
        return y
    
class skip_light(torch.nn.Module):
    """
        implementation of the skip network as a lighter version of the skip_dnn, based on Fady's implementation
        argument:
            config: the configurations file
    """
    def __init__(self, config):
        super(skip_light, self).__init__()
        self.input_shape = config["input_shape"]
        self.width = config["width"]
        self.n_modules = config["depth"]
        self.skip_depth = config["skip_block_layers"]
        self.output_shape = config['n_targets']
        
        # input layer
        self.input = torch.nn.Linear(self.input_shape, self.width)
        torch.nn.init.xavier_normal_(self.input.weight)  # Equivalent to 'glorot_normal'
        torch.nn.init.zeros_(self.input.bias)  # Equivalent to 'zeros'
        self.act = getActivation(config)
        #skip blocks
        self.skip_core = torch.nn.Sequential(*[
            skip_light_module(config) 
            for _ in range(self.n_modules)
        ])
        #output layer
        self.output = torch.nn.Linear(self.width, self.output_shape)
        torch.nn.init.xavier_normal_(self.output.weight)  # Equivalent to 'glorot_normal'
        torch.nn.init.zeros_(self.output.bias)  # Equivalent to 'zeros'
        
    def forward(self, x):
        x = self.act(self.input(x))
        x = self.skip_core(x)
        x = self.output(x)
        return x
    
    
def nets(config):
    """ the pytorch model builder
        arguments:
            config: the configuration file
        returns:
            regressor: the pytorch model
    """
    
    # define the torch model
    if config["model_type"] == 'dnn':
        regressor = dnn(config).double().to(get_device())
    elif config["model_type"] == 'skip':
        regressor = skip_dnn(skip_block, config).double().to(get_device())
    elif config["model_type"] == 'skip-stream':
        regressor = skip_dnn(skip_block, config, stream = True).double().to(get_device())
    elif config["model_type"] == 'skip-light':
        regressor = skip_light(config).double().to(get_device())
    else:
        logging.error(' '+config["model_type"]+' not implemented. model_type can be either dnn, skip or squeeze')
        
        
    # save parameter counts
    summary = pms.summary(regressor, torch.zeros((config["input_shape"],)).to(get_device()).double().clone().detach().requires_grad_(True)).rstrip().split('\n')
    config["trainable_parameters"] = int(summary[-3].replace(',', '')[18:])
    config["non_trainable_parameters"] = int(summary[-2].replace(',', '')[22:])
    config["total_parameters"] = int(summary[-4].replace(',', '')[14:])
    
    # save config
    with open(config['directory']+'/config-'+config['model-uuid']+'.json', 'w') as f:
        json.dump(config, f, indent=4)
        
    return regressor



## Load Model

In [4]:
model_path = "../models/skip-light-torch-4D-y5_y6-ef8e7636--10-31-leaky_relu-1024-adam-exp-schedule-mse-val_mse-97.017682-99.836771/dnn-10-31-leaky_relu-adam-exp-schedule-mse-val_mse.torch" #"../models/run1/skip-torch-4D-y5-55960c0a--16-32-leaky_relu-512-adam-exp-schedule-mse-val_mse-94.928634-99.670260/dnn-16-32-leaky_relu-adam-exp-schedule-mse-val_mse.torch" #"../run1/" + config['directory'] + f'/dnn-{config["depth"]}-{config["width"]}-{config["activation"]}-adam-{config["lr_decay_type"]}-schedule-{config["loss"]}-{config["monitor"]}.torch'

In [33]:
"""models/run1/skip-torch-4D-y5-55960c0a--16-32-leaky_relu-512-adam-exp-schedule-mse-val_mse-94.928634-99.670260/dnn-16-32-leaky_relu-adam-exp-schedule-mse-val_mse.torch"""

'models/run1/skip-torch-4D-y5-55960c0a--16-32-leaky_relu-512-adam-exp-schedule-mse-val_mse-94.928634-99.670260/dnn-16-32-leaky_relu-adam-exp-schedule-mse-val_mse.torch'

In [5]:
!ls {model_path}

../models/skip-light-torch-4D-y5_y6-ef8e7636--10-31-leaky_relu-1024-adam-exp-schedule-mse-val_mse-97.017682-99.836771/dnn-10-31-leaky_relu-adam-exp-schedule-mse-val_mse.torch


In [25]:
# define the torch model
if config["model_type"] == 'dnn':
    regressor = dnn(config).double().to(get_device())
elif config["model_type"] == 'skip':
    regressor = skip_dnn(skip_block, config).double().to(get_device())
elif config["model_type"] == 'skip-stream':
    regressor = skip_dnn(skip_block, config, stream = True).double().to(get_device())
elif config["model_type"] == 'skip-light':
    regressor = skip_light(config).double().to(get_device())
else:
    logging.error(' '+config["model_type"]+' not implemented. model_type can be either dnn, skip or squeeze')

In [28]:
regressor.load_state_dict(torch.load(model_path, map_location=torch.device('cpu')))

/tmp/ipykernel_2387537/1709965805.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  regressor.load_state_dict(torch.load(model_path, map_location=torch.device('cpu')))


<All keys matched successfully>

In [44]:
abs_score, r2_score = test_model(regressor, test_loader, config)

In [45]:
print(' relative accuracy: {:.2f}%  |---|  R2 score: {:.2f}%'.format(abs_score, r2_score))

 relative accuracy: 97.02%  |---|  R2 score: 99.84%


In [46]:
def test_model(model, test_data, config):
    ''' function for testing the model
        arguments:
            model: the pytorch model to nbe trained
            test_data: the test data loader
            config: the configurations file
        returns:
            the absolute error
            the R2 score
    '''
    
    for i, data in enumerate(test_data):
        
        # forward prop
        x, y = data
        y_p = model(x)
        
        # store data
        if i == 0:
            x_test = x.cpu().detach().numpy()
            y_test = y.cpu().detach().numpy()
            y_pred = y_p.cpu().detach().numpy()
        else: 
            x_test = np.vstack((x_test, x.cpu().detach().numpy()))
            y_test = np.vstack((y_test, y.cpu().detach().numpy()))
            y_pred = np.vstack((y_pred, y_p.cpu().detach().numpy()))
        
    
    # accuracy
    y_test = np.array(y_test).reshape(-1, config['n_targets'])
    y_pred = np.array(y_pred).reshape(-1, config['n_targets'])
    abs_score = (1 - np.mean(np.abs((y_pred - y_test)/y_test)))*100
    r2_score = metrics.r2_score(y_test, y_pred)*100
    
    # save test results
    df_pred = pd.DataFrame(x_test, columns=['x'+str(i+1) for i in range(config['input_shape'])])
    
    # unscale the target
    scaled_cname = [s + '_scaled' for s in config['var_y']]
    df_pred = pd.concat([df_pred, pd.DataFrame(y_test, columns=scaled_cname)], axis=1)
    for col in df_pred.columns[4:]:
        df_pred[col[:-7]] = df_pred[col].apply(lambda y: y_unscale(y))
    # y_test_real = pd.DataFrame(np.vectorize(y_unscale)(y_test), columns=config['var_y']) # 
    # df_pred = pd.concat([df_pred, y_test_real], axis=1)
    y_test_real = df_pred.iloc[:,-config['n_targets']:].to_numpy()
    
    # stuff in the scaled predictions
    pred_cname = [s + '_scaled_pred' for s in config['var_y']]
    df_pred = pd.concat([df_pred, pd.DataFrame(y_pred, columns=pred_cname)], axis=1)
    
    # stuff in the unscaled predictions
    pred_cname = [s + '_pred' for s in config['var_y']]
    for col in df_pred.columns[-config['n_targets']:]:
        df_pred[col[:-12]+'_pred'] = df_pred[col].apply(lambda y: y_unscale(y))
    # y_pred_real = pd.DataFrame(np.vectorize(y_unscale)(y_pred), columns=pred_cname) # .apply(lambda y: y_unscale(y))
    # df_pred = pd.concat([df_pred, y_pred_real], axis=1)
    y_pred_real = df_pred.iloc[:,-config['n_targets']:].to_numpy()
    
    # calculate the deltas for the scaled and unscaled targets
    scaled_delta_cname = ['scaled_delta_' + s for s in config['var_y']]
    df_pred = pd.concat([df_pred, pd.DataFrame((y_pred - y_test)/y_test*100, columns=scaled_delta_cname)], axis=1)
    delta_cname = ['delta_' + s for s in config['var_y']]
    df_pred = pd.concat([df_pred, pd.DataFrame((y_pred_real - y_test_real)/y_test_real*100, columns=delta_cname)], axis=1)
    
    #same the whole dataframe
    #df_pred.to_csv(config['directory']+'/test-results-'+config['model-uuid']+'-'+f'{abs_score:.6f}-{r2_score:.6f}.csv')
    
    # plot all the scaled and real deltas
    """for c in scaled_delta_cname:
        make_error_plot(config, df_pred, col=c)
    for c in delta_cname:
        make_error_plot(config, df_pred, col=c)"""
        
    config['test_metrics'] = {}
    config['test_metrics']['r2'] = {}
    config['test_metrics']['abs_score'] = {}
    num_vars = len(config['var_y'])
    base_length = config['input_shape'] + num_vars
    for i in range(base_length, base_length + num_vars):
        test = df_pred.iloc[:,i].values
        pred = df_pred.iloc[:,i + 2 * num_vars].values
        config['test_metrics']['r2'][config['var_y'][i - base_length]] = metrics.r2_score(test, pred)*100
        config['test_metrics']['abs_score'][config['var_y'][i - base_length]] = 100 - np.abs(df_pred.iloc[:,i + 4 * num_vars].mean())
    
    config['test_metrics']['r2']['model'] = r2_score
    config['test_metrics']['abs_score']['model'] = abs_score
    
    return abs_score, r2_score

In [47]:
df_pred

NameError: name 'df_pred' is not defined

In [57]:
df_pred

,x1,x2,x3,x4,y5_scaled,y5,y5_scaled_pred,y5_pred,scaled_delta_y5,delta_y5
0,5.434361e-07,0.147431,0.774351,0.245984,-5.167388,-174.455943,-5.089776,-161.353522,-1.501954,-7.510447
1,2.254383e-06,0.705441,0.511854,0.034134,5.032969,152.387746,5.045020,154.247369,0.239437,1.220323
2,3.135050e-06,0.348233,0.460954,0.283262,4.168857,63.641536,4.144576,62.090847,-0.582449,-2.436600
3,3.604113e-06,0.659904,0.123490,0.852356,5.413599,223.437810,5.399942,220.393467,-0.252275,-1.362501
4,3.774481e-06,0.547915,0.649524,0.693039,5.148154,171.113443,5.128244,167.720558,-0.386740,-1.982828
...,...,...,...,...,...,...,...,...,...,...
499707,3.458530e-01,0.084713,0.169534,0.451104,-4.115073,-60.256685,-4.112973,-60.128185,-0.051030,-0.213255
499708,3.460583e-01,0.315792,0.713225,0.709775,-2.779589,-15.112394,-2.777749,-15.082775,-0.066196,-0.195992
499709,3.465390e-01,0.609794,0.212700,0.336352,-2.043512,-6.717664,-2.033212,-6.638579,-0.504036,-1.177258
499710,3.465490e-01,0.432198,0.130247,0.729713,-2.497521,-11.152331,-2.464341,-10.755735,-1.328509,-3.556170


In [69]:
y_pred

array([[-5.08977619],
       [ 5.04501978],
       [ 4.1445757 ],
       ...,
       [-2.03321163],
       [-2.4643412 ],
       [-2.39796909]])

In [70]:
test

NameError: name 'test' is not defined

In [48]:
model, test_data = regressor, test_loader

In [49]:
regressor.train(False)

skip_light(
  (input): Linear(in_features=4, out_features=31, bias=True)
  (act): LeakyReLU(negative_slope=0.01)
  (skip_core): Sequential(
    (0): skip_light_module(
      (act): LeakyReLU(negative_slope=0.01)
      (fc_module): ModuleList(
        (0-4): 5 x Linear(in_features=31, out_features=31, bias=True)
      )
    )
    (1): skip_light_module(
      (act): LeakyReLU(negative_slope=0.01)
      (fc_module): ModuleList(
        (0-4): 5 x Linear(in_features=31, out_features=31, bias=True)
      )
    )
    (2): skip_light_module(
      (act): LeakyReLU(negative_slope=0.01)
      (fc_module): ModuleList(
        (0-4): 5 x Linear(in_features=31, out_features=31, bias=True)
      )
    )
    (3): skip_light_module(
      (act): LeakyReLU(negative_slope=0.01)
      (fc_module): ModuleList(
        (0-4): 5 x Linear(in_features=31, out_features=31, bias=True)
      )
    )
    (4): skip_light_module(
      (act): LeakyReLU(negative_slope=0.01)
      (fc_module): ModuleList(
        (

In [181]:
for i, data in enumerate(test_data):
    # forward prop
    x, y = data
    y_p = model(x)
    
    # store data
    if i == 0:
        x_test = x.cpu().detach().numpy()
        y_test = y.cpu().detach().numpy()
        y_pred = y_p.cpu().detach().numpy()
    else: 
        x_test = np.vstack((x_test, x.cpu().detach().numpy()))
        y_test = np.vstack((y_test, y.cpu().detach().numpy()))
        y_pred = np.vstack((y_pred, y_p.cpu().detach().numpy()))

In [197]:
(1 - np.mean(np.abs((y_unscaler(y_pred) - y_unscaler(y_test))/y_unscaler(y_test))))*100

93.8972764152744

In [186]:
y_pred

array([[-5.15054851,  5.22246609],
       [ 5.00127931, -5.06639124],
       [ 4.16731525, -4.17138413],
       ...,
       [-2.02787016, -1.29885781],
       [-2.50983625, -1.50882085],
       [-2.41709313, -2.52511211]])

In [189]:
def y_unscaler(y):
    ''' function for scaling x1
        argument:
            y: the input variable
        returns:
            the scaled variable
    '''
    y = np.array(y)  # Ensure input is a NumPy array
    return np.where(y >= 0, np.exp(y) - 1, 1 - np.exp(-y))

In [190]:
y_unscaler(y_pred)

array([[-171.52609724,  184.39081056],
       [ 147.60314683, -157.60094054],
       [  63.54194063,  -63.80508853],
       ...,
       [  -6.59788685,   -2.66510802],
       [ -11.30291523,   -3.52139624],
       [ -10.21321656,  -11.49229566]])

In [185]:
y_unscale(y_pred.values)

AttributeError: 'numpy.ndarray' object has no attribute 'values'

In [90]:
y_test

array([[-5.16738797,  5.23624844],
       [ 5.03296901, -5.0693969 ],
       [ 4.16885718, -4.18429983],
       ...,
       [-2.04351166, -1.28735805],
       [-2.49752098, -1.4918922 ],
       [-2.39736605, -2.49740627]])

In [106]:
a = 0.1
b = 0.000001
np.abs(a-b)/a

0.99999

In [85]:
# accuracy
y_test = np.array(y_test).reshape(-1, config['n_targets'])
y_pred = np.array(y_pred).reshape(-1, config['n_targets'])
abs_score = (1 - np.mean(np.abs((y_pred - y_test)/y_test)))*100
r2_score = metrics.r2_score(y_test, y_pred)*100

In [82]:
np.abs((y_pred - y_test)/y_test)

array([[0.0032588 , 0.0026321 ],
       [0.00629642, 0.0005929 ],
       [0.00036987, 0.00308671],
       ...,
       [0.00765423, 0.00893284],
       [0.00493099, 0.0113471 ],
       [0.00822865, 0.01109385]])

In [83]:
y_pred

array([[-5.15054851,  5.22246609],
       [ 5.00127931, -5.06639124],
       [ 4.16731525, -4.17138413],
       ...,
       [-2.02787016, -1.29885781],
       [-2.50983625, -1.50882085],
       [-2.41709313, -2.52511211]])

In [115]:
# save test results
df_pred = pd.DataFrame(x_test, columns=['x'+str(i+1) for i in range(config['input_shape'])])

In [116]:
df_pred

,x1,x2,x3,x4
0,5.434361e-07,0.147431,0.774351,0.245984
1,2.254383e-06,0.705441,0.511854,0.034134
2,3.135050e-06,0.348233,0.460954,0.283262
3,3.604113e-06,0.659904,0.123490,0.852356
4,3.774481e-06,0.547915,0.649524,0.693039
...,...,...,...,...
499707,3.458530e-01,0.084713,0.169534,0.451104
499708,3.460583e-01,0.315792,0.713225,0.709775
499709,3.465390e-01,0.609794,0.212700,0.336352
499710,3.465490e-01,0.432198,0.130247,0.729713


In [117]:
# unscale the target
scaled_cname = [s + '_scaled' for s in config['var_y']]
df_pred = pd.concat([df_pred, pd.DataFrame(y_test, columns=scaled_cname)], axis=1)

In [118]:
df_pred

,x1,x2,x3,x4,y5_scaled,y6_scaled
0,5.434361e-07,0.147431,0.774351,0.245984,-5.167388,5.236248
1,2.254383e-06,0.705441,0.511854,0.034134,5.032969,-5.069397
2,3.135050e-06,0.348233,0.460954,0.283262,4.168857,-4.184300
3,3.604113e-06,0.659904,0.123490,0.852356,5.413599,-5.481205
4,3.774481e-06,0.547915,0.649524,0.693039,5.148154,-5.184982
...,...,...,...,...,...,...
499707,3.458530e-01,0.084713,0.169534,0.451104,-4.115073,0.524211
499708,3.460583e-01,0.315792,0.713225,0.709775,-2.779589,-1.419106
499709,3.465390e-01,0.609794,0.212700,0.336352,-2.043512,-1.287358
499710,3.465490e-01,0.432198,0.130247,0.729713,-2.497521,-1.491892


In [122]:
for col in df_pred.columns[4:]:
    df_pred[col[:-7]] = df_pred[col].apply(lambda y: y_unscale(y))

In [126]:
df_pred

,x1,x2,x3,x4,y5_scaled,y6_scaled,y5,y6
0,5.434361e-07,0.147431,0.774351,0.245984,-5.167388,5.236248,-174.455943,186.963621
1,2.254383e-06,0.705441,0.511854,0.034134,5.032969,-5.069397,152.387746,-158.078358
2,3.135050e-06,0.348233,0.460954,0.283262,4.168857,-4.184300,63.641536,-64.647520
3,3.604113e-06,0.659904,0.123490,0.852356,5.413599,-5.481205,223.437810,-239.135900
4,3.774481e-06,0.547915,0.649524,0.693039,5.148154,-5.184982,171.113443,-177.570199
...,...,...,...,...,...,...,...,...
499707,3.458530e-01,0.084713,0.169534,0.451104,-4.115073,0.524211,-60.256685,0.689126
499708,3.460583e-01,0.315792,0.713225,0.709775,-2.779589,-1.419106,-15.112394,-3.133422
499709,3.465390e-01,0.609794,0.212700,0.336352,-2.043512,-1.287358,-6.717664,-2.623202
499710,3.465490e-01,0.432198,0.130247,0.729713,-2.497521,-1.491892,-11.152331,-3.445499


In [134]:
# y_test_real = pd.DataFrame(np.vectorize(y_unscale)(y_test), columns=config['var_y']) # 
# df_pred = pd.concat([df_pred, y_test_real], axis=1)
y_test_real = df_pred.iloc[:,-config['n_targets']:].to_numpy()

# stuff in the scaled predictions
pred_cname = [s + '_scaled_pred' for s in config['var_y']]
df_pred = pd.concat([df_pred, pd.DataFrame(y_pred, columns=pred_cname)], axis=1)

# stuff in the unscaled predictions
pred_cname = [s + '_pred' for s in config['var_y']]
for col in df_pred.columns[-config['n_targets']:]:
    df_pred[col[:-12]+'_pred'] = df_pred[col].apply(lambda y: y_unscale(y))
# y_pred_real = pd.DataFrame(np.vectorize(y_unscale)(y_pred), columns=pred_cname) # .apply(lambda y: y_unscale(y))
# df_pred = pd.concat([df_pred, y_pred_real], axis=1)
y_pred_real = df_pred.iloc[:,-config['n_targets']:].to_numpy()

# calculate the deltas for the scaled and unscaled targets
scaled_delta_cname = ['scaled_delta_' + s for s in config['var_y']]
df_pred = pd.concat([df_pred, pd.DataFrame((y_pred - y_test)/y_test*100, columns=scaled_delta_cname)], axis=1)
delta_cname = ['delta_' + s for s in config['var_y']]
df_pred = pd.concat([df_pred, pd.DataFrame((y_pred_real - y_test_real)/y_test_real*100, columns=delta_cname)], axis=1)

In [207]:
y_pred_real

array([[-171.52609724,  184.39081056],
       [ 147.60314683, -157.60094054],
       [  63.54194063,  -63.80508853],
       ...,
       [  -6.59788685,   -2.66510802],
       [ -11.30291523,   -3.52139624],
       [ -10.21321656,  -11.49229566]])

In [208]:
y_test_real

array([[-174.45594286,  186.96362132],
       [ 152.38774629, -158.07835759],
       [  63.64153637,  -64.64752027],
       ...,
       [  -6.71766351,   -2.62320157],
       [ -11.15233077,   -3.44549933],
       [  -9.99418008,  -11.15093677]])

In [225]:
(1 - np.mean(np.abs((y_pred_real - y_test_real)/y_test_real)))*100

93.8972764152744

In [226]:
deltas = (y_pred_real - y_test_real)/y_test_real*100
deltas

array([[-1.67941864, -1.37610233],
       [-3.13975341, -0.30201291],
       [-0.15649488, -1.30311533],
       ...,
       [-1.78301077,  1.59753055],
       [ 1.35025102,  2.20278397],
       [ 2.19164033,  3.06125756]])

In [223]:
100 - np.abs(df_pred.iloc[:,i + 4 * num_vars].mean())

99.65569012719484

In [224]:
df_pred.iloc[:,i + 4 * num_vars]

0         -1.376102
1         -0.302013
2         -1.303115
3          0.361828
4          1.120664
            ...    
499707   -14.290330
499708    -1.085019
499709     1.597531
499710     2.202784
499711     3.061258
Name: delta_y6, Length: 499712, dtype: float64

In [237]:
100 - np.abs((np.mean(deltas)))

99.90727647574951

In [231]:
(deltas[:, 1])

array([-1.37610233, -0.30201291, -1.30311533, ...,  1.59753055,
        2.20278397,  3.06125756])

In [210]:
y_pred

array([[-5.15054851,  5.22246609],
       [ 5.00127931, -5.06639124],
       [ 4.16731525, -4.17138413],
       ...,
       [-2.02787016, -1.29885781],
       [-2.50983625, -1.50882085],
       [-2.41709313, -2.52511211]])

In [ ]:
(y_pred_real - y_test_real)/y_test_real*100

In [213]:
y_test_real

array([[-174.45594286,  186.96362132],
       [ 152.38774629, -158.07835759],
       [  63.64153637,  -64.64752027],
       ...,
       [  -6.71766351,   -2.62320157],
       [ -11.15233077,   -3.44549933],
       [  -9.99418008,  -11.15093677]])

In [158]:
df_pred

,x1,x2,x3,x4,y5_scaled,y6_scaled,y5,y6,y5_scaled_pred,y6_scaled_pred,y5_pred,y6_pred,scaled_delta_y5,scaled_delta_y6,delta_y5,delta_y6
0,5.434361e-07,0.147431,0.774351,0.245984,-5.167388,5.236248,-174.455943,186.963621,-5.150549,5.222466,-171.526097,184.390811,-0.325880,-0.263210,-1.679419,-1.376102
1,2.254383e-06,0.705441,0.511854,0.034134,5.032969,-5.069397,152.387746,-158.078358,5.001279,-5.066391,147.603147,-157.600941,-0.629642,-0.059290,-3.139753,-0.302013
2,3.135050e-06,0.348233,0.460954,0.283262,4.168857,-4.184300,63.641536,-64.647520,4.167315,-4.171384,63.541941,-63.805089,-0.036987,-0.308671,-0.156495,-1.303115
3,3.604113e-06,0.659904,0.123490,0.852356,5.413599,-5.481205,223.437810,-239.135900,5.367740,-5.484802,213.377844,-240.001161,-0.847100,0.065619,-4.502356,0.361828
4,3.774481e-06,0.547915,0.649524,0.693039,5.148154,-5.184982,171.113443,-177.570199,5.124916,-5.196064,167.160015,-179.560164,-0.451382,0.213737,-2.310414,1.120664
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
499707,3.458530e-01,0.084713,0.169534,0.451104,-4.115073,0.524211,-60.256685,0.689126,-4.090355,0.464141,-58.761115,0.590647,-0.600665,-11.459121,-2.481999,-14.290330
499708,3.460583e-01,0.315792,0.713225,0.709775,-2.779589,-1.419106,-15.112394,-3.133422,-2.804856,-1.410846,-15.524700,-3.099423,0.909036,-0.582001,2.728267,-1.085019
499709,3.465390e-01,0.609794,0.212700,0.336352,-2.043512,-1.287358,-6.717664,-2.623202,-2.027870,-1.298858,-6.597887,-2.665108,-0.765423,0.893284,-1.783011,1.597531
499710,3.465490e-01,0.432198,0.130247,0.729713,-2.497521,-1.491892,-11.152331,-3.445499,-2.509836,-1.508821,-11.302915,-3.521396,0.493099,1.134710,1.350251,2.202784


In [63]:
import copy
cfg = copy.deepcopy(config)

In [ ]:
# accuracy
y_test = np.array(y_test).reshape(-1, config['n_targets'])
y_pred = np.array(y_pred).reshape(-1, config['n_targets'])
abs_score = (1 - np.mean(np.abs((y_pred - y_test)/y_test)))*100
r2_score = metrics.r2_score(y_test, y_pred)*100

In [166]:
y_pred[:, 0]

array([-5.15054851,  5.00127931,  4.16731525, ..., -2.02787016,
       -2.50983625, -2.41709313])

In [171]:
metrics.r2_score(y_test, y_pred)*100

99.8367708984802

In [169]:
metrics.r2_score(y_test[:, 0], y_pred[:, 0])*100

99.80237053943921

In [170]:
metrics.r2_score(y_test[:, 1], y_pred[:, 1])*100

99.87117125752117

In [180]:
y_pred_real, y_test_real

(array([[-171.52609724,  184.39081056],
        [ 147.60314683, -157.60094054],
        [  63.54194063,  -63.80508853],
        ...,
        [  -6.59788685,   -2.66510802],
        [ -11.30291523,   -3.52139624],
        [ -10.21321656,  -11.49229566]]),
 array([[-174.45594286,  186.96362132],
        [ 152.38774629, -158.07835759],
        [  63.64153637,  -64.64752027],
        ...,
        [  -6.71766351,   -2.62320157],
        [ -11.15233077,   -3.44549933],
        [  -9.99418008,  -11.15093677]]))

In [199]:
cfg['test_metrics'] = {}
cfg['test_metrics']['r2'] = {}
cfg['test_metrics']['abs_score'] = {}
num_vars = len(cfg['var_y'])
base_length = cfg['input_shape'] + num_vars
for i in range(base_length, base_length + num_vars):
    test = df_pred.iloc[:,i].values
    pred = df_pred.iloc[:,i + 2 * num_vars].values
    cfg['test_metrics']['r2'][cfg['var_y'][i - base_length]] = metrics.r2_score(test, pred)*100
    cfg['test_metrics']['abs_score'][cfg['var_y'][i - base_length]] = 100 - np.abs(df_pred.iloc[:,i + 4 * num_vars].mean())

cfg['test_metrics']['r2']['model'] = r2_score
cfg['test_metrics']['abs_score']['model'] = abs_score

In [206]:
df_pred.iloc[:,i + 4 * num_vars]

0         -1.376102
1         -0.302013
2         -1.303115
3          0.361828
4          1.120664
            ...    
499707   -14.290330
499708    -1.085019
499709     1.597531
499710     2.202784
499711     3.061258
Name: delta_y6, Length: 499712, dtype: float64

In [202]:
metrics.r2_score(test, pred)*100

41.53556692441735

In [162]:
1-np.abs(df_pred.iloc[:,i + 4 * num_vars].mean())

0.6556901271948501

In [222]:
cfg['test_metrics']

{'r2': {'y5': 59.349493799409636,
  'y6': 41.53556692441735,
  'model': 99.8367708984802},
 'abs_score': {'y5': 99.47024307869387,
  'y6': 99.65569012719484,
  'model': 97.01768156003149}}

In [151]:
df_pred

,x1,x2,x3,x4,y5_scaled,y6_scaled,y5,y6,y5_scaled_pred,y6_scaled_pred,y5_pred,y6_pred,scaled_delta_y5,scaled_delta_y6,delta_y5,delta_y6
0,5.434361e-07,0.147431,0.774351,0.245984,-5.167388,5.236248,-174.455943,186.963621,-5.150549,5.222466,-171.526097,184.390811,-0.325880,-0.263210,-1.679419,-1.376102
1,2.254383e-06,0.705441,0.511854,0.034134,5.032969,-5.069397,152.387746,-158.078358,5.001279,-5.066391,147.603147,-157.600941,-0.629642,-0.059290,-3.139753,-0.302013
2,3.135050e-06,0.348233,0.460954,0.283262,4.168857,-4.184300,63.641536,-64.647520,4.167315,-4.171384,63.541941,-63.805089,-0.036987,-0.308671,-0.156495,-1.303115
3,3.604113e-06,0.659904,0.123490,0.852356,5.413599,-5.481205,223.437810,-239.135900,5.367740,-5.484802,213.377844,-240.001161,-0.847100,0.065619,-4.502356,0.361828
4,3.774481e-06,0.547915,0.649524,0.693039,5.148154,-5.184982,171.113443,-177.570199,5.124916,-5.196064,167.160015,-179.560164,-0.451382,0.213737,-2.310414,1.120664
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
499707,3.458530e-01,0.084713,0.169534,0.451104,-4.115073,0.524211,-60.256685,0.689126,-4.090355,0.464141,-58.761115,0.590647,-0.600665,-11.459121,-2.481999,-14.290330
499708,3.460583e-01,0.315792,0.713225,0.709775,-2.779589,-1.419106,-15.112394,-3.133422,-2.804856,-1.410846,-15.524700,-3.099423,0.909036,-0.582001,2.728267,-1.085019
499709,3.465390e-01,0.609794,0.212700,0.336352,-2.043512,-1.287358,-6.717664,-2.623202,-2.027870,-1.298858,-6.597887,-2.665108,-0.765423,0.893284,-1.783011,1.597531
499710,3.465490e-01,0.432198,0.130247,0.729713,-2.497521,-1.491892,-11.152331,-3.445499,-2.509836,-1.508821,-11.302915,-3.521396,0.493099,1.134710,1.350251,2.202784


In [140]:
for i in range(base_length, base_length + num_vars):
    print(i)

6
7


In [138]:
cfg

{'model_type': 'skip-light',
 'input_shape': 4,
 'amplitude_shape': 36,
 'data_dir': '/scratch/akula.ha/dataset',
 'seed': 42,
 'var_y': ['y5', 'y6'],
 'activation': 'leaky_relu',
 'width': 31,
 'depth': 10,
 'skip_block_layers': 5,
 'beta': 0,
 'alpha': 0,
 'normal_scaled': False,
 'lr_decay_type': 'exp',
 'initial_lr': 0.003,
 'final_lr': 1e-06,
 'decay_steps': 120000,
 'train-sample-size': 10000000,
 'validate-sample-size': 500000,
 'test-sample-size': 500000,
 'use_MC_sample': False,
 'batch_size': 1024,
 'steps_per_epoch': 2400,
 'early_stopping_start_epoch': 50,
 'patience': 50,
 'monitor': 'val_mse',
 'loss': 'mse',
 'gradient_clipping': True,
 'verbose': 1,
 'base_directory': '../models/',
 'epochs': 2000,
 'model-uuid': 'ef8e7636',
 'start_time': '2025-05-07 21:39:40 -0400',
 'device': 'cuda',
 'directory': '../models/skip-light-torch-4D-y5_y6-ef8e7636',
 'train-samples-used': 6,
 'validate-samples-used': 6,
 'test-samples-used': 6,
 'n_targets': 2,
 'scaling': {},
 'trainable

In [69]:
r2_score = metrics.r2_score(y_test, y_pred)*100
abs_score = (1 - np.mean(np.abs((y_pred - y_test)/y_test)))*100

In [70]:
r2_score, abs_score

(99.8367708984802, 97.01768156003149)

In [71]:
y_test, y_pred

(array([[-5.16738797,  5.23624844],
        [ 5.03296901, -5.0693969 ],
        [ 4.16885718, -4.18429983],
        ...,
        [-2.04351166, -1.28735805],
        [-2.49752098, -1.4918922 ],
        [-2.39736605, -2.49740627]]),
 array([[-5.15054851,  5.22246609],
        [ 5.00127931, -5.06639124],
        [ 4.16731525, -4.17138413],
        ...,
        [-2.02787016, -1.29885781],
        [-2.50983625, -1.50882085],
        [-2.41709313, -2.52511211]]))

In [72]:
metrics.r2_score(y_test, y_pred)

0.998367708984802

In [73]:
df_pred.iloc[:,i + 4 * num_vars]

0         -1.376102
1         -0.302013
2         -1.303115
3          0.361828
4          1.120664
            ...    
499707   -14.290330
499708    -1.085019
499709     1.597531
499710     2.202784
499711     3.061258
Name: delta_y6, Length: 499712, dtype: float64

In [74]:
100 - np.abs(df_pred.iloc[:,i + 4 * num_vars].mean())

99.65569012719484

In [75]:
test, pred

(array([ 186.96362132, -158.07835759,  -64.64752027, ...,   -2.62320157,
          -3.44549933,  -11.15093677]),
 array([ 184.39081056, -157.60094054,  -63.80508853, ...,   -2.66510802,
          -3.52139624,  -11.49229566]))